# 01 — Data Cleaning
## Smart Expense Intelligence Dashboard
**Goal:** Load raw transaction data, inspect it, fix all issues, and export a clean dataset ready for EDA and SQL.

In [6]:
import pandas as pd
import numpy as np
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Libraries loaded ✓")

Libraries loaded ✓


In [7]:
df = pd.read_csv(r"E:\Projects\Smart Expense Intelligence Dashboard\data\raw_transactions.csv")

print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

Shape: (50000, 9)
Rows: 50,000 | Columns: 9


,Customer ID,Name,Surname,Gender,Birthdate,Transaction Amount,Date,Merchant Name,Category
0,752858,Sean,Rodriguez,F,20-10-2002,35.47,03-04-2023,Smith-Russell,Cosmetic
1,26381,Michelle,Phelps,NaN,24-10-1985,2552.72,17-07-2023,"Peck, Spence and Young",Travel
2,305449,Jacob,Williams,M,25-10-1981,115.97,20-09-2023,Steele Inc,Clothing
3,988259,Nathan,Snyder,M,26-10-1977,11.31,11-01-2023,"Wilson, Wilson and Russell",Cosmetic
4,764762,Crystal,Knapp,F,02-11-1951,62.21,13-06-2023,Palmer-Hinton,Electronics


In [8]:
print("=== DATA TYPES ===")
print(df.dtypes)
print("\n=== NULL VALUES ===")
print(df.isnull().sum())
print("\n=== DUPLICATES ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

=== DATA TYPES ===
Customer ID             int64
Name                      str
Surname                   str
Gender                    str
Birthdate                 str
Transaction Amount    float64
Date                      str
Merchant Name             str
Category                  str
dtype: object

=== NULL VALUES ===
Customer ID              0
Name                     0
Surname                  0
Gender                5047
Birthdate                0
Transaction Amount       0
Date                     0
Merchant Name            0
Category                 0
dtype: int64

=== DUPLICATES ===
Duplicate rows: 0


## Step 1 — Fix Column Names
Strip spaces and standardise to snake_case

In [10]:
# Clean column names — lowercase, replace spaces with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("Cleaned column names:")
print(df.columns.tolist())

Cleaned column names:
['customer_id', 'name', 'surname', 'gender', 'birthdate', 'transaction_amount', 'date', 'merchant_name', 'category']


## Step 2 — Fix Date Columns
`date` and `birthdate` are strings — convert both to datetime

In [11]:
# Fix transaction date
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')

# Fix birthdate
df['birthdate'] = pd.to_datetime(df['birthdate'], dayfirst=True, errors='coerce')

# Verify
print(df[['date', 'birthdate']].dtypes)
print(f"\nDate nulls after conversion: {df['date'].isnull().sum()}")
print(f"Birthdate nulls after conversion: {df['birthdate'].isnull().sum()}")

date         datetime64[us]
birthdate    datetime64[us]
dtype: object

Date nulls after conversion: 0
Birthdate nulls after conversion: 0


## Step 3 — Extract Date Features
These columns will be critical for SQL queries and Power BI slicers

In [13]:
df['year']        = df['date'].dt.year
df['month']       = df['date'].dt.month
df['month_name']  = df['date'].dt.strftime('%B')
df['quarter']     = df['date'].dt.quarter
df['week_number'] = df['date'].dt.isocalendar().week.astype(int)
df['day_of_week'] = df['date'].dt.strftime('%A')
df['is_weekend']  = df['date'].dt.dayofweek.isin([5, 6]).astype(int)

# Fixed age calculation — handles NaT birthdates safely
df['age'] = ((df['date'] - df['birthdate']).dt.days / 365.25)
df['age'] = pd.to_numeric(df['age'], errors='coerce').round(0).astype('float64')

print("New columns added successfully")
print(df[['date','year','month','month_name','quarter',
          'week_number','day_of_week','is_weekend','age']].head())

New columns added successfully
        date  year  month month_name  quarter  week_number day_of_week  \
0 2023-04-03  2023      4      April        2           14      Monday   
1 2023-07-17  2023      7       July        3           29      Monday   
2 2023-09-20  2023      9  September        3           38   Wednesday   
3 2023-01-11  2023      1    January        1            2   Wednesday   
4 2023-06-13  2023      6       June        2           24     Tuesday   

   is_weekend   age  
0           0 20.00  
1           0 38.00  
2           0 42.00  
3           0 45.00  
4           0 72.00  


## Step 4 — Handle Missing Values
Only `gender` has 1,047 nulls (~2%). Fill with 'Unknown' — do NOT drop rows.

In [14]:
print(f"Null counts before:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

# Fill gender nulls
df['gender'] = df['gender'].fillna('Unknown')

# Standardise gender values
df['gender'] = df['gender'].str.strip().str.capitalize()
print(f"\nGender value counts:\n{df['gender'].value_counts()}")

print(f"\nNull counts after:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n✅ No nulls remaining" if df.isnull().sum().sum() == 0 else f"⚠️ Nulls still exist: {df.isnull().sum().sum()}")

Null counts before:
gender    5047
dtype: int64

Gender value counts:
gender
F          22713
M          22240
Unknown     5047
Name: count, dtype: int64

Null counts after:
Series([], dtype: int64)

✅ No nulls remaining


In [15]:
print("=== TRANSACTION AMOUNT STATS ===")
print(df['transaction_amount'].describe())

# Check for zero or negative amounts
zero_neg = df[df['transaction_amount'] <= 0]
print(f"\nZero or negative transactions: {len(zero_neg)}")

# Flag outliers using IQR
Q1 = df['transaction_amount'].quantile(0.25)
Q3 = df['transaction_amount'].quantile(0.75)
IQR = Q3 - Q1
outlier_threshold = Q3 + (3 * IQR)

df['is_outlier'] = (df['transaction_amount'] > outlier_threshold).astype(int)
print(f"\nOutlier threshold: ₹{outlier_threshold:,.2f}")
print(f"Outlier transactions flagged: {df['is_outlier'].sum()}")

=== TRANSACTION AMOUNT STATS ===
count   50000.00
mean      442.12
std       631.67
min         5.01
25%        79.01
50%       182.19
75%       470.51
max      2999.88
Name: transaction_amount, dtype: float64

Zero or negative transactions: 0

Outlier threshold: ₹1,645.04
Outlier transactions flagged: 3873


In [16]:
# Standardise category
df['category'] = df['category'].str.strip().str.title()
print("Categories found:")
print(df['category'].value_counts())

# Standardise merchant name
df['merchant_name'] = df['merchant_name'].str.strip().str.title()
print(f"\nUnique merchants: {df['merchant_name'].nunique()}")

Categories found:
category
Restaurant     8413
Market         8382
Travel         8377
Electronics    8324
Clothing       8261
Cosmetic       8243
Name: count, dtype: int64

Unique merchants: 36939


In [17]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Duplicates removed: {before - after}")

Rows before: 50,000
Rows after:  50,000
Duplicates removed: 0


In [18]:
print("=== FINAL DATASET SUMMARY ===")
print(f"Shape         : {df.shape}")
print(f"Nulls         : {df.isnull().sum().sum()}")
print(f"Duplicates    : {df.duplicated().sum()}")
print(f"Date range    : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Categories    : {df['category'].nunique()} → {df['category'].unique().tolist()}")
print(f"Avg amount    : ₹{df['transaction_amount'].mean():,.2f}")
print(f"Total spend   : ₹{df['transaction_amount'].sum():,.2f}")
print(f"Outliers flagged : {df['is_outlier'].sum()}")

df.head(3)

=== FINAL DATASET SUMMARY ===
Shape         : (50000, 18)
Nulls         : 0
Duplicates    : 0
Date range    : 2023-01-01 → 2023-10-14
Categories    : 6 → ['Cosmetic', 'Travel', 'Clothing', 'Electronics', 'Restaurant', 'Market']
Avg amount    : ₹442.12
Total spend   : ₹22,105,961.97
Outliers flagged : 3873


,customer_id,name,surname,gender,birthdate,transaction_amount,date,merchant_name,category,year,month,month_name,quarter,week_number,day_of_week,is_weekend,age,is_outlier
0,752858,Sean,Rodriguez,F,2002-10-20,35.47,2023-04-03,Smith-Russell,Cosmetic,2023,4,April,2,14,Monday,0,20.00,0
1,26381,Michelle,Phelps,Unknown,1985-10-24,2552.72,2023-07-17,"Peck, Spence And Young",Travel,2023,7,July,3,29,Monday,0,38.00,1
2,305449,Jacob,Williams,M,1981-10-25,115.97,2023-09-20,Steele Inc,Clothing,2023,9,September,3,38,Wednesday,0,42.00,0


In [19]:
df.head()

,customer_id,name,surname,gender,birthdate,transaction_amount,date,merchant_name,category,year,month,month_name,quarter,week_number,day_of_week,is_weekend,age,is_outlier
0,752858,Sean,Rodriguez,F,2002-10-20,35.47,2023-04-03,Smith-Russell,Cosmetic,2023,4,April,2,14,Monday,0,20.00,0
1,26381,Michelle,Phelps,Unknown,1985-10-24,2552.72,2023-07-17,"Peck, Spence And Young",Travel,2023,7,July,3,29,Monday,0,38.00,1
2,305449,Jacob,Williams,M,1981-10-25,115.97,2023-09-20,Steele Inc,Clothing,2023,9,September,3,38,Wednesday,0,42.00,0
3,988259,Nathan,Snyder,M,1977-10-26,11.31,2023-01-11,"Wilson, Wilson And Russell",Cosmetic,2023,1,January,1,2,Wednesday,0,45.00,0
4,764762,Crystal,Knapp,F,1951-11-02,62.21,2023-06-13,Palmer-Hinton,Electronics,2023,6,June,2,24,Tuesday,0,72.00,0


In [20]:
df.to_csv("output.csv", index=False)

In [21]:
output_path = '../data/clean_transactions.csv'
df.to_csv(output_path, index=False)

print(f"✅ Clean data saved → {output_path}")
print(f"   Rows: {len(df):,} | Columns: {len(df.columns)}")
print(f"\nFinal columns: {df.columns.tolist()}")

✅ Clean data saved → ../data/clean_transactions.csv
   Rows: 50,000 | Columns: 18

Final columns: ['customer_id', 'name', 'surname', 'gender', 'birthdate', 'transaction_amount', 'date', 'merchant_name', 'category', 'year', 'month', 'month_name', 'quarter', 'week_number', 'day_of_week', 'is_weekend', 'age', 'is_outlier']
